In [14]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import madina as md
import madina.una.tools as una
import folium
from folium.plugins import MarkerCluster, Draw
from folium.plugins import Draw, MarkerCluster
from branca.colormap import linear

In [2]:
os.chdir('/workspaces/exp_sidewalk_planning/')
from src.funcs import betweeness

Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   357 | /workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson
buildings            |       1 | EPSG:3857  |  2860 | /workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson
kitas                |       1 | EPSG:3857  |    80 | /workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson
Geographic center: (13.396303813961305, 52.4916423315232)
No network graph yet. First, insert a layer that contains network segments (streets, sidewalks, ..) and call create_street_network(layer_name,  weight_attribute=None)
	Then,  insert origins and destinations using 'insert_nodes(label, layer_name, weight_attribute)'
	Finally, when done, create a network by calling 'create_street_network()'


Skipping field time: unsupported OGR type: 10


In [ ]:
### load base model
berlin = md.Zonal()
berlin.load_layer(
        name='sidewalks',
        source='/workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson'
    )

berlin.load_layer('buildings', '/workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson')
berlin.load_layer('kitas', '/workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson')

berlin.create_street_network(source_layer="sidewalks", node_snapping_tolerance=0.1)

# Default nodes
berlin.insert_node(label='origin', layer_name="kitas")
berlin.insert_node(label='destination', layer_name="buildings")
berlin.create_graph()

berlin.describe()

Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   357 | /workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson
buildings            |       1 | EPSG:3857  |  2860 | /workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson
kitas                |       1 | EPSG:3857  |    80 | /workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson
Geographic center: (13.396303813961305, 52.4916423315232)
Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   357 | /workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson
buildings            |       1 | EPSG:3857  |  2860 | /workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson
kitas                |       1 | EPSG:3857  |    80 | /workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson
Geographic center: (13.396303813961305, 52.4916423

In [16]:
result_gdf = betweeness(
    zonal_obj = berlin,
    origin='kitas',
    destination='buildings',
    job_id=1,
    between_radius=1000,
    alpha=0.1
)      

INFO:root:Graph created


In [22]:
result_gdf.columns

Index(['element_nr', 'strassensc', 'strassenna', 'str_bez', 'strassenkl',
       'strassen_1', 'strassen_2', 'verkehrsri', 'bezirk', 'stadtteil',
       'verkehrseb', 'beginnt_be', 'endet_bei_', 'laenge', 'gueltig_vo',
       'okstra_id', 'geometry', 'weight', 'betweenness'],
      dtype='object')

In [23]:
cols = ['strassenna','bezirk', 'geometry', 'weight', 'betweenness']
result_gdf = result_gdf[cols]

In [25]:
result_gdf.head(2)

,strassenna,bezirk,geometry,weight,betweenness
id,,,,,
0,Arndtstraße,Friedrichshain-Kreuzberg,"LINESTRING (1490867.213 6888927.4, 1491060.883...",196.033187,0.000022
1,Baerwaldstraße,Friedrichshain-Kreuzberg,"LINESTRING (1491906.88 6889679.628, 1491922.38...",280.495739,0.011419


In [28]:
gdf_wgs = result_gdf.to_crs(epsg=4326)
gdf_wgs

,strassenna,bezirk,geometry,weight,betweenness
id,,,,,
0,Arndtstraße,Friedrichshain-Kreuzberg,"LINESTRING (13.39269 52.48844, 13.39443 52.48827)",196.033187,2.232140e-05
1,Baerwaldstraße,Friedrichshain-Kreuzberg,"LINESTRING (13.40203 52.49255, 13.40217 52.492...",280.495739,1.141885e-02
2,Baerwaldstraße,Friedrichshain-Kreuzberg,"LINESTRING (13.40317 52.49392, 13.4042 52.49526)",270.777566,4.454925e-14
3,Baerwaldstraße,Friedrichshain-Kreuzberg,"LINESTRING (13.4042 52.49526, 13.40502 52.49631)",212.854732,2.286939e-37
4,Bergmannstraße,Friedrichshain-Kreuzberg,"LINESTRING (13.39011 52.48962, 13.392 52.48936)",215.344639,1.910027e+00
...,...,...,...,...,...
352,Möckernkiez,Friedrichshain-Kreuzberg,"LINESTRING (13.3781 52.49279, 13.37867 52.49294)",68.745932,4.816544e-17
353,Möckernkiez,Friedrichshain-Kreuzberg,"LINESTRING (13.37891 52.49268, 13.37867 52.49294)",54.498437,4.202042e-17
354,Möckernkiez,Friedrichshain-Kreuzberg,"LINESTRING (13.37867 52.49294, 13.37876 52.493...",116.126929,7.024152e-18


In [34]:
gdf = gdf_wgs

In [38]:
center = gdf.unary_union.centroid
home_location = [center.y, center.x]

# Create color scale
colormap = linear.Reds_09.scale(gdf['betweenness'].min(), gdf['betweenness'].max())
colormap.caption = 'Betweenness Centrality'

# Function to style each feature
def style_function(feature):
    value = feature['properties']['betweenness']
    return {
        'color': colormap(value),
        'weight': 3,
        'opacity': 0.8
    }

# Create map
map_0 = folium.Map(location=home_location, zoom_start=15, tiles="cartodb positron")
Draw(draw_options={'polyline': False, 'marker': False, 'circlemarker': False}).add_to(map_0)
marker_cluster = MarkerCluster().add_to(map_0)

# Add GeoDataFrame to map
folium.GeoJson(
    gdf_wgs,    
    name='Betweenness',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["strassenna", "bezirk", "betweenness"]),
).add_to(map_0)

# Add legend
colormap.add_to(map_0)

# Show map (if using Streamlit, wrap this in st_folium)
map_0

/tmp/ipykernel_81669/2688870460.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf.unary_union.centroid


### study area

In [ ]:
### load base model
berlin = md.Zonal()
berlin.load_layer(
        name='sidewalks',
        source='/workspaces/exp_sidewalk_planning/data/study_area/sidewalk.geojson'
    )

berlin.load_layer('buildings', '/workspaces/exp_sidewalk_planning/data/study_area/buildings.geojson')
berlin.load_layer('kitas', '/workspaces/exp_sidewalk_planning/data/study_area/kitas.geojson')
berlin.load_layer('grundschulen', '/workspaces/exp_sidewalk_planning/data/study_area/grundschule.geojson')
berlin.load_layer('gymnasien', '/workspaces/exp_sidewalk_planning/data/study_area/gymnasium.geojson')
berlin.load_layer('cafes', '/workspaces/exp_sidewalk_planning/data/study_area/cafes.geojson')

berlin.create_street_network(source_layer="sidewalks", node_snapping_tolerance=0.1)

In [102]:
berlin.create_graph()
berlin.describe()

Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   947 | /workspaces/exp_sidewalk_planning/data/study_area/sidewalk.geojson
buildings            |       1 | EPSG:3857  |  5928 | /workspaces/exp_sidewalk_planning/data/study_area/buildings.geojson
kitas                |       1 | EPSG:3857  |   212 | /workspaces/exp_sidewalk_planning/data/study_area/kitas.geojson
grundschulen         |       1 | EPSG:3857  |    23 | /workspaces/exp_sidewalk_planning/data/study_area/grundschule.geojson
gymnasien            |       1 | EPSG:3857  |     4 | /workspaces/exp_sidewalk_planning/data/study_area/gymnasium.geojson
cafes                |       1 | EPSG:3857  |   307 | /workspaces/exp_sidewalk_planning/data/study_area/cafes.geojson
home_us              |       1 | EPSG:3857  |     1 | /workspaces/exp_sidewalk_planning/data/study_area/home_us.geojson
home_oma             |       1 | EPSG:3857  |     1 | /workspaces/exp_sidewalk_p

In [34]:
def betweeness(origin:str, destination:str, job_id:int, between_radius: int, detour_ratio:int = 1.2, beta:float = 0.001) -> list:
    
    berlin.clear_nodes()
    berlin.insert_node(label='origin', layer_name=origin)
    berlin.insert_node(label='destination', layer_name=destination)
    berlin.create_graph()

    una.betweenness(
        berlin,
        search_radius = between_radius,
        detour_ratio=1.2,
        decay=True,
        decay_method="exponent",
        beta=beta,
        turn_penalty=True,
        closest_destination=False,
        elastic_weight = True,
        knn_weight = [0.5, 0.25, 0.25],
        knn_plateau = 400,
        save_betweenness_as="betweenness",
        num_cores=8
    )
    
    gdf = berlin['sidewalks'].gdf   
       
    return gdf

In [114]:
gdf = betweeness('home_us', 'home_oma', 1, 9000, 0, 0)

In [115]:
gdf

,element_nr,strassensc,strassenna,str_bez,strassenkl,strassen_1,strassen_2,verkehrsri,bezirk,stadtteil,verkehrseb,beginnt_be,endet_bei_,laenge,gueltig_vo,okstra_id,geometry,weight,betweenness
id,,,,,,,,,,,,,,,,,,,
0,46520003_46520004.01,00394,Baerwaldstraße,None,III,G,STRA,B,Friedrichshain-Kreuzberg,Kreuzberg,0,46520003,46520004,129.5700,2010-01-01,2653E5785AC146DC8839BFE3281F64FC,"LINESTRING (1492149.161 6890174.449, 1492239.9...",212.854732,0.059280
1,45520004_45520005.01,00613,Blücherstraße,None,II,G,STRA,B,Friedrichshain-Kreuzberg,Kreuzberg,0,45520004,45520005,82.4800,2010-01-01,6F802D167E544207BD6D7302DD536A4D,"LINESTRING (1490509.829 6890309.573, 1490535.4...",135.229074,0.090069
2,45520005_46520006.01,00613,Blücherstraße,None,II,G,STRA,B,Friedrichshain-Kreuzberg,Kreuzberg,0,45520005,46520006,303.0400,2010-01-01,3829A16E527C455E9E7E2F697975C525,"LINESTRING (1490643.841 6890294.319, 1490653.9...",496.847035,0.090069
3,45520012_46520024.01,01561,Gitschiner Straße,None,II,G,STRA,G,Friedrichshain-Kreuzberg,Kreuzberg,0,45520012,46520024,266.4400,2010-01-01,5B9EBAAEECFB4F71AFD7A25AB1D2E84A,"LINESTRING (1490787.212 6890666.151, 1490817.0...",436.880588,0.046169
4,46520026_46520027.01,01561,Gitschiner Straße,None,II,G,STRA,B,Friedrichshain-Kreuzberg,Kreuzberg,0,46520026,46520027,245.2600,2010-01-01,B6B3140E3CBF48318C896140D8C2675F,"LINESTRING (1492015.822 6890743.524, 1492404.6...",402.118597,0.091182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
942,48500020_48500038.01,01445,Fuldastraße,None,V,G,STRA,B,Neukölln,Neukölln,0,48500020,48500038,131.8791,2016-07-26,7427816E041D429FA9B174177AE6C9F4,"LINESTRING (1495541.235 6887907.471, 1495557.3...",216.485153,0.000000
943,49510016_49510030.01,08292,Weichselpark,None,V,F,PARK,B,Neukölln,Neukölln,0,49510016,49510030,128.2681,2016-07-26,AB8E2560A3804E4A8E870432F8EA674C,"LINESTRING (1496052.729 6889042.642, 1496068.0...",210.738348,0.000000
944,48510025_48500004.01,02922,Mainzer Straße,None,V,G,STRA,B,Neukölln,Neukölln,0,48510025,48500004,303.5920,2017-09-04,59FBD47D804140FAA4F518E99A21CE9C,"LINESTRING (1494736.864 6888287.913, 1494736.3...",32.084393,0.000000


In [116]:
cols = ['strassenna', 'bezirk', 'stadtteil','geometry', 'weight', 'betweenness']

In [117]:
gdf = gdf[cols]

In [129]:
center = gdf_wgs.unary_union.centroid
home_location = [center.y, center.x]

# Create color scale
#colormap = linear.Reds_09.scale(gdf_wgs['betweenness'].min(), gdf_wgs['betweenness'].max())
colormap.caption = 'Betweenness Centrality'

# Function to style each feature
def style_function(feature):
    value = feature['properties']['betweenness']
    return {
        'color': colormap(value),
        'weight': 3,
        #'opacity': 0.8
    }

/tmp/ipykernel_427/1230538424.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf_wgs.unary_union.centroid


In [130]:
map_0 = folium.Map(location=home_location, zoom_start=15, tiles="cartodb positron")
Draw(draw_options={'polyline': False, 'marker': False, 'circlemarker': False}).add_to(map_0)
marker_cluster = MarkerCluster().add_to(map_0)

folium.GeoJson(
    gdf_wgs,    
    name='Betweenness',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["strassenna", "bezirk", "betweenness"]),
).add_to(map_0)

# Add legend
colormap.add_to(map_0)

# Show map (if using Streamlit, wrap this in st_folium)
map_0